## Active Learning dataset creation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# --- 1. CONFIGURATION ---

# A. ORIGINAL UNIVERSE (Your current Train/Val/Test)
# These files define the IDs used in your existing splits
ORIG_MOL_DF = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df.pkl"
ORIG_SPEC_DF = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df.pkl"
SPLIT_DIR = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits"

In [30]:
# --- 2. HELPER: MAP BUILDER ---
def build_lookup_maps(mol_df_path, spec_df_path, name="Dataset"):
    print(f"\n--- Building Lookups for {name} ---")
    
    # 1. Mol ID -> Scaffold
    print(f"Loading Molecules: {mol_df_path}")
    df_mols = pd.read_pickle(mol_df_path)
    # Ensure scaffold column exists
    if 'scaffold' not in df_mols.columns:
        raise ValueError(f"'{mol_df_path}' is missing the 'scaffold' column!")
    
    mol2scaffold = dict(zip(df_mols['mol_id'], df_mols['scaffold']))
    
    # 2. Spec ID -> Mol ID & Precursor m/z
    print(f"Loading Spectra: {spec_df_path}")
    df_specs = pd.read_pickle(spec_df_path)
    
    spec2mol = dict(zip(df_specs['spec_id'], df_specs['mol_id']))
    spec2mz = dict(zip(df_specs['spec_id'], df_specs['prec_mz']))
    
    print(f" > {name} Ready: {len(spec2mol)} spectra, {len(mol2scaffold)} molecules.")
    return spec2mol, mol2scaffold, spec2mz

In [31]:
# --- 3. BUILD ORIGINAL FIREWALL ---
# We use the ORIGINAL maps to decode the Validation/Test sets
orig_spec2mol, orig_mol2scaffold, orig_spec2mz = build_lookup_maps(ORIG_MOL_DF, ORIG_SPEC_DF, "ORIGINAL DB")


--- Building Lookups for ORIGINAL DB ---
Loading Molecules: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df.pkl
Loading Spectra: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df.pkl
 > ORIGINAL DB Ready: 227308 spectra, 27969 molecules.


In [32]:
# --- CONFIGURATION ---
ORIGINAL_TRAIN_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train.feather"
SAFE_CANDIDATES_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/mona_safe_augmentation_candidates.feather"
OUTPUT_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train.feather"

In [33]:
# Target Strategy: "MAX" matches the largest bin. "FIXED" sets a specific number (e.g., 20k).
STRATEGY = "MAX" 
FIXED_LIMIT = 50000 # Only used if STRATEGY = "FIXED"

In [34]:
# --- 1. LOAD DATA ---
print("--- Loading Datasets ---")
df_train = pd.read_feather(ORIGINAL_TRAIN_PATH)
df_safe = pd.read_feather(SAFE_CANDIDATES_PATH)

print(f"Original Train Size: {len(df_train)}")
print(f"Safe Candidates Available: {len(df_safe)}")

--- Loading Datasets ---
Original Train Size: 179998
Safe Candidates Available: 635053


In [35]:
# Calculate Mass Diff using MONA m/z data
if 'mass_difference' not in df_train.columns:
    print("Calculating mass differences...")
    mz_a = df_train['name_main'].map(orig_spec2mz)
    mz_b = df_train['name_sub'].map(orig_spec2mz)
    df_train['mass_difference'] = abs(mz_a - mz_b)
    # Filter out NaNs if any lookups failed
    df_train = df_train.dropna(subset=['mass_difference'])

Calculating mass differences...


In [36]:
# --- 2. DEFINE REGIMES (Helper) ---
# Ensure Mass_Regime column exists in original train
def assign_regime(row):
    # Use existing column if available, else calc
    if 'Mass_Regime' in row and not pd.isna(row['Mass_Regime']):
        return row['Mass_Regime']
    
    md = row.get('mass_difference')
    if md is None: return "Unknown"
    
    if md < 0.01: return "0. Exact Isomers (0 Da)"
    if md < 1.0: return "1. Isobaric (< 1 Da)"
    if md < 10.0: return "2. Tiny (1-10 Da)"
    if md < 50.0: return "3. Medium (10-50 Da)"
    if md < 100.0: return "4. Large (50-100 Da)"
    return "5. Huge (>100 Da)"

In [37]:
if 'Mass_Regime' not in df_train.columns:
    print("Calculating Mass Regimes for Original Train...")
    df_train['Mass_Regime'] = df_train.apply(assign_regime, axis=1)

Calculating Mass Regimes for Original Train...


In [38]:
df_train.head()

,name_main,name_sub,tanimoto,cosine_similarity,label,mass_difference,Mass_Regime
0,MassSpecGymID0078974,MassSpecGymID0214701,0.188679,0.702941,1.0,132.02454,5. Huge (>100 Da)
1,MassSpecGymID0074719,MassSpecGymID0218883,0.148148,0.621023,0.0,66.05147,4. Large (50-100 Da)
2,MassSpecGymID0007593,MassSpecGymID0224784,0.538462,0.951866,1.0,193.99977,5. Huge (>100 Da)
3,MassSpecGymID0064493,MassSpecGymID0065607,0.416667,0.510570,0.0,24.03640,3. Medium (10-50 Da)
4,MassSpecGymID0034253,MassSpecGymID0039027,0.200000,0.357671,0.0,19.08740,3. Medium (10-50 Da)


In [39]:
# Ensure Mass_Regime format matches in df_safe (Clean up strings if needed)
# The previous script output "3. Medium (10-50 Da)", so we normalize if needed.
# For now, we assume simple string matching works or we map them.
# Let's normalize to short names just in case:
short_names = {
    "0. Exact Isomers (0 Da)": "0. Exact Isomers",
    "1. Isobaric (< 1 Da)": "1. Isobaric",
    "2. Tiny (1-10 Da)": "2. Tiny",
    "3. Medium (10-50 Da)": "3. Medium",
    "4. Large (50-100 Da)": "4. Large", 
    "5. Huge (>100 Da)": "5. Huge"
}
# A robust normalizer function
def normalize_regime(val):
    for key in short_names:
        if key in str(val): return key
    return "Unknown"

df_train['Regime_Norm'] = df_train['Mass_Regime'].apply(normalize_regime)
df_safe['Regime_Norm'] = df_safe['Mass_Regime'].apply(normalize_regime)

In [40]:
# --- 3. ANALYZE GAPS ---
print("\n--- Current Distribution (Original Train) ---")
counts = df_train['Regime_Norm'].value_counts().sort_index()
print(counts)

if STRATEGY == "MAX":
    target_count = counts.max()
else:
    target_count = FIXED_LIMIT

print(f"\nTarget per Regime: {target_count}")


--- Current Distribution (Original Train) ---
Regime_Norm
0. Exact Isomers (0 Da)    10302
1. Isobaric (< 1 Da)        2203
2. Tiny (1-10 Da)           9429
3. Medium (10-50 Da)       50746
4. Large (50-100 Da)       36317
5. Huge (>100 Da)          70984
Name: count, dtype: int64

Target per Regime: 70984


In [41]:
# --- 4. AUGMENTATION LOOP ---
print("\n--- Starting Augmentation ---")
augmented_dfs = [df_train] # Start with original data

for regime in counts.index:
    current_n = counts[regime]
    deficit = target_count - current_n
    
    if deficit <= 0:
        print(f" > {regime}: OK (Current {current_n} >= Target {target_count})")
        continue
        
    # We need more!
    print(f" > {regime}: Needs +{deficit} pairs...")

    # Filter candidates
    candidates = df_safe[df_safe['Regime_Norm'] == regime]
    available = len(candidates)
    
    if available == 0:
        print(f"   !!! Warning: No candidates available for {regime}. Skipping.")
        continue
        
    # Sample
    n_sample = min(deficit, available)
    subset = candidates.sample(n=n_sample, random_state=42)
    
    # Align columns (Safe candidates might have extra/fewer columns)
    # We select only the cols present in original train to keep it clean
    common_cols = list(set(df_train.columns) & set(subset.columns))
    subset = subset[common_cols]
    
    augmented_dfs.append(subset)
    print(f"   -> Added {n_sample} pairs. (Coverage: {n_sample/deficit*100:.1f}%)")


--- Starting Augmentation ---
 > 0. Exact Isomers (0 Da): Needs +60682 pairs...
   -> Added 5518 pairs. (Coverage: 9.1%)
 > 1. Isobaric (< 1 Da): Needs +68781 pairs...
   -> Added 4987 pairs. (Coverage: 7.3%)
 > 2. Tiny (1-10 Da): Needs +61555 pairs...
   -> Added 30243 pairs. (Coverage: 49.1%)
 > 3. Medium (10-50 Da): Needs +20238 pairs...
   -> Added 20238 pairs. (Coverage: 100.0%)
 > 4. Large (50-100 Da): Needs +34667 pairs...
   -> Added 34667 pairs. (Coverage: 100.0%)
 > 5. Huge (>100 Da): OK (Current 70984 >= Target 70984)


In [45]:
# --- 5. COMBINE AND SAVE ---
print("\n--- Finalizing Dataset ---")
df_final = pd.concat(augmented_dfs, axis=0, ignore_index=True)

# Shuffle
df_final = df_final.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Original Size: {len(df_train)}")
print(f"Augmented Size: {len(df_final)}")
print(f"Increase: +{len(df_final) - len(df_train)} pairs")

# Final Distribution Check
print("\n--- Final Distribution ---")
print(df_final['Regime_Norm'].value_counts().sort_index())

# Save
df_final.to_feather(OUTPUT_PATH)
print(f"\nSaved Balanced Dataset to: {OUTPUT_PATH}")


--- Finalizing Dataset ---
Original Size: 179981
Augmented Size: 275634
Increase: +95653 pairs

--- Final Distribution ---
Regime_Norm
0. Exact Isomers (0 Da)    15820
1. Isobaric (< 1 Da)        7190
2. Tiny (1-10 Da)          39672
3. Medium (10-50 Da)       70984
4. Large (50-100 Da)       70984
5. Huge (>100 Da)          70984
Name: count, dtype: int64

Saved Balanced Dataset to: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train.feather


In [1]:
import pandas as pd
import numpy as np
import os

In [4]:
# --- CONFIGURATION ---
# Point this to the file that caused the training crash
BROKEN_DATASET_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train.feather"

# The output path for the fixed file
FIXED_DATASET_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train_FIXED.feather"

# Your Match Threshold (Must match what you used for the original data)
THRESHOLD = 0.7

In [5]:
# --- 1. LOAD ---
print(f"Loading dataset: {BROKEN_DATASET_PATH}")
df = pd.read_feather(BROKEN_DATASET_PATH)
print(f"Total Rows: {len(df)}")

Loading dataset: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train.feather
Total Rows: 275634


In [6]:
# --- 2. DIAGNOSE ---
# Check if 'label' column exists
if 'label' not in df.columns:
    print(" > 'label' column MISSING entirely. Creating it now...")
    missing_mask = pd.Series([True] * len(df)) # All rows need labels
else:
    # Check for NaNs
    missing_mask = df['label'].isna()
    nan_count = missing_mask.sum()
    print(f" > Found {nan_count} rows with NaN labels (Likely the MoNA pairs).")
    
    if nan_count == 0:
        print(" > Weird... No NaN labels found. Is the column already fixed?")

 > Found 95653 rows with NaN labels (Likely the MoNA pairs).


In [7]:
# --- 3. PATCH ---
if missing_mask.sum() > 0:
    print(f"--- Patching {missing_mask.sum()} labels (Threshold: {THRESHOLD}) ---")
    
    # Vectorized calculation: fast and efficient
    # 1 if sim >= 0.7, else 0
    new_labels = (df.loc[missing_mask, 'cosine_similarity'] >= THRESHOLD).astype(int)
    
    # Assign back
    df.loc[missing_mask, 'label'] = new_labels
    
    # Verification
    print(f" > Patch applied.")
    print(f" > New Label Distribution in Patched Rows:")
    print(df.loc[missing_mask, 'label'].value_counts())

--- Patching 95653 labels (Threshold: 0.7) ---
 > Patch applied.
 > New Label Distribution in Patched Rows:
label
0.0    57103
1.0    38550
Name: count, dtype: int64


In [8]:
# --- 4. FINAL CHECK & SAVE ---
# Double check for any remaining NaNs
if df['label'].isna().sum() > 0:
    print("!!! CRITICAL WARNING: NaNs still exist. Something went wrong.")
else:
    print(" > Integrity Check Passed: 0 NaNs.")

# Ensure Integer type (floats like 1.0 can cause issues in some loss functions)
df['label'] = df['label'].astype(int)

print(f"Saving fixed dataset to: {FIXED_DATASET_PATH}")
df.to_feather(FIXED_DATASET_PATH)
print("Done! You can update your training script path and restart.")

 > Integrity Check Passed: 0 NaNs.
Saving fixed dataset to: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train_FIXED.feather
Done! You can update your training script path and restart.


## Restore Original Dataset

In [50]:
import pandas as pd
import os

# --- CONFIGURATION ---
# Path to the current augmented/modified file
AUGMENTED_FILE_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train_FIXED.feather"

# Path where you want to save the restored original file
RESTORED_OUTPUT_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train.feather"

In [51]:
# --- 1. LOAD DATA ---
print(f"Loading augmented dataset: {AUGMENTED_FILE_PATH}")
df = pd.read_feather(AUGMENTED_FILE_PATH)
print(f"Current Size: {len(df)} pairs")

Loading augmented dataset: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train_FIXED.feather
Current Size: 275634 pairs


In [52]:
# --- 2. IDENTIFY MONA PAIRS ---
# We look for rows where EITHER name_main OR name_sub starts with "MoNA_"
# This covers cases where a MoNA spectrum might have been paired with a non-MoNA one (unlikely in your workflow, but safer).
print("\n--- Identifying MoNA Pairs ---")

mask_mona = df['name_main'].astype(str).str.startswith("MoNA_") | \
            df['name_sub'].astype(str).str.startswith("MoNA_")

n_mona = mask_mona.sum()
n_original = (~mask_mona).sum()

print(f" > Found {n_mona} MoNA pairs to remove.")
print(f" > Found {n_original} Original pairs to keep.")


--- Identifying MoNA Pairs ---
 > Found 95653 MoNA pairs to remove.
 > Found 179981 Original pairs to keep.


In [53]:
# --- 3. FILTER AND VERIFY ---
print("\n--- Removing Added Pairs ---")
df_restored = df[~mask_mona].copy()

# Integrity Check
# Ensure no MoNA IDs remain
remaining_mona = df_restored['name_main'].astype(str).str.startswith("MoNA_").sum()
if remaining_mona > 0:
    print(f"!!! WARNING: {remaining_mona} MoNA pairs still remain. Check filter logic.")
else:
    print(" > Success: All MoNA pairs removed.")


--- Removing Added Pairs ---
 > Success: All MoNA pairs removed.


In [54]:
# --- 4. SAVE ---
print(f"\nSaving restored dataset to: {RESTORED_OUTPUT_PATH}")
df_restored.reset_index(drop=True).to_feather(RESTORED_OUTPUT_PATH)
print("Done.")


Saving restored dataset to: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train.feather
Done.
